# Freeze and Release Reproducibility Audit

**Objective.** Verify the split-training release, downstream freeze validators, stress gates, and artifact hashes.

**Run mode.** Analysis only. This notebook reads locked ORIUS artifacts
and does not retrain models, rewrite release manifests, or mutate runtime traces.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = Path.cwd().parent
PUBLICATION = ROOT / "reports" / "publication"
SPLIT_ROOT = ROOT / "reports" / "split_training"
RELEASE_ID = (SPLIT_ROOT / "latest_release_id.txt").read_text().strip()
FREEZE = ROOT / "reports" / "predeployment_freeze" / RELEASE_ID

def read_csv(relpath: str) -> pd.DataFrame:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def read_json(relpath: str) -> dict:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())

def display_path(relpath: str) -> None:
    path = ROOT / relpath
    print(f"{relpath}: {'exists' if path.exists() else 'missing'}")

## Release manifest and hashes

In [ ]:
model_gate = read_json(f"reports/split_training/{RELEASE_ID}/candidate_model_quality_gate.json")
downstream = read_json(f"reports/predeployment_freeze/{RELEASE_ID}/downstream_post_split_results.json")
runtime_stress = read_json(f"reports/predeployment_freeze/{RELEASE_ID}/final_runtime_stress_gates_post_split.json")
manifest = read_json(f"reports/predeployment_freeze/{RELEASE_ID}/predeployment_release_manifest.json")
hashes = read_json(f"reports/predeployment_freeze/{RELEASE_ID}/frozen_artifact_hashes.json")

summary = {
    "release_id": RELEASE_ID,
    "model_gate_pass": model_gate["pass"],
    "release_models": sum(1 for row in model_gate["models"] if row.get("release_model")),
    "model_blockers": len(model_gate["blockers"]),
    "downstream_all_passed": downstream["all_passed"],
    "downstream_validators": len(downstream["results"]),
    "runtime_stress_all_passed": runtime_stress["all_passed"],
    "manifest_all_passed": manifest["all_passed"],
    "hashed_artifacts": len(hashes["artifacts"]),
}
pd.DataFrame([summary])

In [ ]:
assert model_gate["pass"]
assert len(model_gate["blockers"]) == 0
assert downstream["all_passed"]
assert runtime_stress["all_passed"]
assert manifest["all_passed"]
assert len(hashes["artifacts"]) > 0